# Двухфазная прокси-модель давления и насыщенности (SimpleFNO)

Переход от однофазной стационарной задачи к двухфазной (нефть+вода) с временной динамикой.

Ключевые отличия от однофазной версии:
- Модель предсказывает ДВА поля: давление И насыщенность водой
- Появилось время: каждый сэмпл — это траектория из 10 шагов
- Временной шаг закодирован как дополнительный входной канал
- Вход: 4 канала (проницаемость + маска нагн. + маска доб. + нормализованное время)
- Выход: 2 канала (давление + насыщенность)

Запускать ячейки строго по порядку сверху вниз.

## 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Импорты и проверка GPU

In [ ]:
import os, glob, time
from collections import defaultdict
import numpy as np
import scipy.io as sio
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Пути

In [ ]:
DATA_DIR   = '/content/drive/MyDrive/proxy_model/data_2phase/'
OUT_DIR    = '/content/drive/MyDrive/proxy_model/'
MODEL_PATH = os.path.join(OUT_DIR, 'best_model_2phase_v1.pt')

files = sorted(glob.glob(os.path.join(DATA_DIR, 'sample_*.mat')))
print('Найдено файлов:', len(files))
assert len(files) > 0, 'Файлы не найдены - проверь DATA_DIR'

## 4. Валидация данных

Проверяем все файлы перед обучением: NaN/Inf, физичность насыщенности [0,1],
наличие реального течения (насыщенность должна меняться во времени).

In [ ]:
import sys

bad_files = []
good_files = []
stats = {'pres_std': [], 'sat_std': [], 'sat_final_mean': [], 'n_steps': []}

for idx, fpath in enumerate(files):
    if idx % 25 == 0:
        print(f'Проверено {idx}/{len(files)}...')
        sys.stdout.flush()

    try:
        d = sio.loadmat(fpath)
    except Exception as e:
        bad_files.append((os.path.basename(fpath), f'не читается: {e}'))
        continue

    pres_traj = d.get('pressure_traj')
    sat_traj  = d.get('saturation_traj')
    perm      = d.get('perm_field')

    if pres_traj is None or sat_traj is None or perm is None:
        bad_files.append((os.path.basename(fpath), 'отсутствует поле'))
        continue

    if np.isnan(pres_traj).any() or np.isinf(pres_traj).any():
        bad_files.append((os.path.basename(fpath), 'NaN/Inf в давлении'))
        continue
    if np.isnan(sat_traj).any() or np.isinf(sat_traj).any():
        bad_files.append((os.path.basename(fpath), 'NaN/Inf в насыщенности'))
        continue

    if sat_traj.min() < -1e-6 or sat_traj.max() > 1 + 1e-6:
        bad_files.append((os.path.basename(fpath),
                          f'насыщенность вне [0,1]: [{sat_traj.min():.3f}, {sat_traj.max():.3f}]'))
        continue

    sat_change = np.abs(sat_traj[:, :, -1] - sat_traj[:, :, 0]).max()
    if sat_change < 0.01:
        bad_files.append((os.path.basename(fpath),
                          f'насыщенность не меняется ({sat_change:.4f})'))
        continue

    good_files.append(fpath)
    stats['pres_std'].append(pres_traj[:, :, -1].std())
    stats['sat_std'].append(sat_traj[:, :, -1].std())
    stats['sat_final_mean'].append(sat_traj[:, :, -1].mean())
    stats['n_steps'].append(sat_traj.shape[2])

print(f'\nВсего: {len(files)} | Годных: {len(good_files)} | Битых: {len(bad_files)}')
if stats['pres_std']:
    print(f'std давления (фин.): {np.mean(stats["pres_std"]):.2f} бар')
    print(f'std насыщенности (фин.): {np.mean(stats["sat_std"]):.4f}')
    print(f'средняя насыщенность (фин.): {np.mean(stats["sat_final_mean"]):.3f}')
    print(f'число временных шагов: {stats["n_steps"][0]}')

if bad_files:
    print('\nПроблемные файлы:')
    for name, reason in bad_files:
        print(f'  {name}: {reason}')

files = good_files
print(f'\nДалее используем {len(files)} годных файлов')

## 5. Визуализация одного сэмпла — эволюция насыщенности и давления

In [ ]:
d = sio.loadmat(files[0])
sat_traj  = d['saturation_traj']
pres_traj = d['pressure_traj']
wells = d['well_coords'].astype(int)
wtype = d['well_type'].astype(int).ravel()

n_steps = sat_traj.shape[2]
show_steps = np.linspace(0, n_steps - 1, min(5, n_steps)).astype(int)

inj_i  = [wells[k, 0] - 1 for k in range(len(wtype)) if wtype[k] == 1]
inj_j  = [wells[k, 1] - 1 for k in range(len(wtype)) if wtype[k] == 1]
prod_i = [wells[k, 0] - 1 for k in range(len(wtype)) if wtype[k] != 1]
prod_j = [wells[k, 1] - 1 for k in range(len(wtype)) if wtype[k] != 1]

fig, axes = plt.subplots(2, len(show_steps), figsize=(4 * len(show_steps), 8))
for col, t in enumerate(show_steps):
    im = axes[0, col].imshow(sat_traj[:, :, t].T, origin='lower', cmap='Blues', vmin=0, vmax=1)
    axes[0, col].scatter(inj_i, inj_j, c='red', s=40, edgecolors='k', linewidths=.6)
    axes[0, col].scatter(prod_i, prod_j, c='lime', s=40, edgecolors='k', linewidths=.6)
    axes[0, col].set_title(f'Sw, шаг {t}')
    plt.colorbar(im, ax=axes[0, col], fraction=.046)

    im2 = axes[1, col].imshow(pres_traj[:, :, t].T, origin='lower', cmap='viridis')
    axes[1, col].scatter(inj_i, inj_j, c='red', s=40, edgecolors='k', linewidths=.6)
    axes[1, col].scatter(prod_i, prod_j, c='lime', s=40, edgecolors='k', linewidths=.6)
    axes[1, col].set_title(f'P, шаг {t}')
    plt.colorbar(im2, ax=axes[1, col], fraction=.046, label='бар')

plt.suptitle('Эволюция насыщенности (верх) и давления (низ)', fontsize=13)
plt.tight_layout(); plt.show()

## 6. Чтение данных и "разворачивание" траекторий

Ключевая идея: каждый временной шаг каждого сэмпла становится **отдельным
обучающим примером**. Сэмпл с 10 временными шагами порождает 10 пар
(вход → выход). Время кодируется как дополнительный канал — постоянное
значение t/t_max по всей сетке 40×40 (от 0 до 1).

200 файлов × 10 шагов = 2000 обучающих пар — примерно столько же,
сколько было в однофазной версии, но каждая пара сложнее.

In [ ]:
WELL_SIGMA = 1.5

_ref = np.zeros((15, 15)); _ref[7, 7] = 1.0
_PEAK = gaussian_filter(_ref, sigma=WELL_SIGMA, mode='constant').max()

def smooth_mask(points, nx, ny):
    m = np.zeros((nx, ny))
    for (i, j) in points:
        m[i, j] += 1.0
    m = gaussian_filter(m, sigma=WELL_SIGMA, mode='constant')
    return m / _PEAK


# --- собираем все развёрнутые примеры ---
all_X = []   # (perm_n, inj, prod, time_channel) -> 4 канала
all_Y_p = [] # давление
all_Y_s = [] # насыщенность
all_pts = [] # координаты скважин для визуализации

# сначала собираем сырые данные для вычисления нормировки
raw_perm, raw_pres, raw_sat = [], [], []

t0 = time.time()
for k, fpath in enumerate(files):
    d = sio.loadmat(fpath)
    raw_perm.append(d['perm_field'])
    raw_pres.append(d['pressure_traj'])
    raw_sat.append(d['saturation_traj'])

raw_perm_all = np.stack(raw_perm)
raw_pres_all = np.concatenate([p.reshape(-1) for p in raw_pres])
raw_sat_all  = np.concatenate([s.reshape(-1) for s in raw_sat])

log_perm = np.log10(raw_perm_all)
NORM = {
    'KMIN': float(log_perm.min()), 'KMAX': float(log_perm.max()),
    'PMIN': float(raw_pres_all.min()), 'PMAX': float(raw_pres_all.max()),
    'SMIN': float(raw_sat_all.min()),  'SMAX': float(raw_sat_all.max()),
    'WELL_SIGMA': WELL_SIGMA,
}
print(f"log10(k): [{NORM['KMIN']:.3f}, {NORM['KMAX']:.3f}]")
print(f"давление: [{NORM['PMIN']:.2f}, {NORM['PMAX']:.2f}] бар")
print(f"насыщенность: [{NORM['SMIN']:.4f}, {NORM['SMAX']:.4f}]")

def mm(a, lo, hi):
    return (a - lo) / (hi - lo) if hi > lo else a * 0

# --- теперь разворачиваем ---
for k, fpath in enumerate(files):
    d = sio.loadmat(fpath)
    perm = d['perm_field']
    pres_traj = d['pressure_traj']
    sat_traj  = d['saturation_traj']
    coords = d['well_coords'].astype(int)
    wtype  = d['well_type'].astype(int).ravel()

    nx, ny = perm.shape
    n_steps = pres_traj.shape[2]

    inj_pts  = [(I-1, J-1) for (I, J), t in zip(coords, wtype) if t == 1]
    prod_pts = [(I-1, J-1) for (I, J), t in zip(coords, wtype) if t != 1]

    perm_n = mm(np.log10(perm), NORM['KMIN'], NORM['KMAX'])
    inj_mask  = smooth_mask(inj_pts, nx, ny)
    prod_mask = smooth_mask(prod_pts, nx, ny)

    for t in range(n_steps):
        t_norm = t / max(n_steps - 1, 1)
        time_ch = np.full((nx, ny), t_norm)

        x = np.stack([perm_n, inj_mask, prod_mask, time_ch], axis=-1)

        p_n = mm(pres_traj[:, :, t], NORM['PMIN'], NORM['PMAX'])
        s_n = mm(sat_traj[:, :, t],  NORM['SMIN'], NORM['SMAX'])

        all_X.append(x)
        all_Y_p.append(p_n)
        all_Y_s.append(s_n)
        all_pts.append((inj_pts, prod_pts))

    if (k + 1) % 50 == 0:
        print(f'развёрнуто {k+1}/{len(files)}')

all_X   = np.stack(all_X).astype(np.float32)     # (N_total, 40, 40, 4)
all_Y_p = np.stack(all_Y_p).astype(np.float32)   # (N_total, 40, 40)
all_Y_s = np.stack(all_Y_s).astype(np.float32)   # (N_total, 40, 40)

print(f'\nготово за {time.time()-t0:.1f} c')
print(f'всего пар вход-выход: {all_X.shape[0]}')
print(f'X: {all_X.shape}  Y_p: {all_Y_p.shape}  Y_s: {all_Y_s.shape}')

## 7. Train/val split и аугментация

In [ ]:
N = all_X.shape[0]
NVAL = max(1, int(round(0.2 * N)))
NTRAIN = N - NVAL

rng = np.random.RandomState(SEED)
idx = rng.permutation(N)
tr_idx, va_idx = idx[:NTRAIN], idx[NTRAIN:]

X_tr, Yp_tr, Ys_tr = all_X[tr_idx], all_Y_p[tr_idx], all_Y_s[tr_idx]
X_va, Yp_va, Ys_va = all_X[va_idx], all_Y_p[va_idx], all_Y_s[va_idx]

print('train:', X_tr.shape[0], ' val:', X_va.shape[0])

# аугментация отражениями
AUGMENT = True
if AUGMENT:
    def flip3(Xa, Ya, Yb):
        Xs = [Xa, np.flip(Xa, axis=1), np.flip(Xa, axis=2),
              np.flip(np.flip(Xa, axis=2), axis=1)]
        Ya_ = [Ya, np.flip(Ya, axis=1), np.flip(Ya, axis=2),
               np.flip(np.flip(Ya, axis=2), axis=1)]
        Yb_ = [Yb, np.flip(Yb, axis=1), np.flip(Yb, axis=2),
               np.flip(np.flip(Yb, axis=2), axis=1)]
        return (np.concatenate(Xs, 0).copy(),
                np.concatenate(Ya_, 0).copy(),
                np.concatenate(Yb_, 0).copy())

    X_tr, Yp_tr, Ys_tr = flip3(X_tr, Yp_tr, Ys_tr)
    print('train после аугментации:', X_tr.shape[0])

X_tr_t  = torch.from_numpy(X_tr).float()
Yp_tr_t = torch.from_numpy(Yp_tr).float()
Ys_tr_t = torch.from_numpy(Ys_tr).float()
X_va_t  = torch.from_numpy(X_va).float()
Yp_va_t = torch.from_numpy(Yp_va).float()
Ys_va_t = torch.from_numpy(Ys_va).float()

BATCH = 10
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_tr_t, Yp_tr_t, Ys_tr_t),
    batch_size=BATCH, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_va_t, Yp_va_t, Ys_va_t),
    batch_size=BATCH, shuffle=False)

## 8. Модель: SimpleFNO с двумя выходами

Та же SpectralConv2d, тот же "ствол" из Fourier-блоков, но на выходе
два раздельных "хвоста" — один для давления, один для насыщенности.
Пока без перекрёстного гейтинга (как в DB-AFNO) — это для следующей итерации.

In [ ]:
IN_CH  = 4    # perm + inj + prod + time
MODES  = 18
WIDTH  = 32
LAYERS = 3


class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, m1, m2):
        super().__init__()
        self.m1, self.m2 = m1, m2
        s = 1.0 / (in_ch * out_ch)
        self.w1 = nn.Parameter(s * torch.rand(in_ch, out_ch, m1, m2, dtype=torch.cfloat))
        self.w2 = nn.Parameter(s * torch.rand(in_ch, out_ch, m1, m2, dtype=torch.cfloat))

    def forward(self, x):
        b, _, nx, ny = x.shape
        xf = torch.fft.rfft2(x)
        o = torch.zeros(b, self.w1.shape[1], nx, ny//2+1, dtype=torch.cfloat, device=x.device)
        o[:,:,:self.m1,:self.m2] = torch.einsum('bixy,ioxy->boxy',
            xf[:,:,:self.m1,:self.m2], self.w1)
        o[:,:,-self.m1:,:self.m2] = torch.einsum('bixy,ioxy->boxy',
            xf[:,:,-self.m1:,:self.m2], self.w2)
        return torch.fft.irfft2(o, s=(nx, ny))


class TwoPhaseFNO(nn.Module):
    """Вход (b, 40, 40, 4) -> два выхода (b, 40, 40) каждый."""
    def __init__(self, in_ch=IN_CH, modes=MODES, width=WIDTH, n_layers=LAYERS):
        super().__init__()
        self.width = width
        self.fc0 = nn.Linear(in_ch, width)

        self.specs = nn.ModuleList([SpectralConv2d(width, width, modes, modes)
                                    for _ in range(n_layers)])
        self.ws = nn.ModuleList([nn.Conv1d(width, width, 1)
                                 for _ in range(n_layers)])

        # два раздельных "хвоста"
        self.head_p = nn.Sequential(nn.Linear(width, 64), nn.ReLU(), nn.Linear(64, 1))
        self.head_s = nn.Sequential(nn.Linear(width, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):
        b, nx, ny, _ = x.shape
        x = self.fc0(x).permute(0, 3, 1, 2)

        for spec, w in zip(self.specs, self.ws):
            x1 = spec(x)
            x2 = w(x.reshape(b, self.width, -1)).reshape(b, self.width, nx, ny)
            x = F.relu(x1 + x2)

        x = x.permute(0, 2, 3, 1)     # (b, nx, ny, width)
        p = self.head_p(x).squeeze(-1) # (b, nx, ny)
        s = self.head_s(x).squeeze(-1)
        return p, s


model = TwoPhaseFNO().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'параметров: {n_params:,}')

## 9. Функция потерь: relative L2 по каждому полю + суммарный loss

In [ ]:
class LpLoss:
    def __init__(self, p=2):
        self.p = p
    def __call__(self, x, y):
        n = x.size(0)
        d = torch.norm(x.reshape(n,-1) - y.reshape(n,-1), self.p, 1)
        b = torch.norm(y.reshape(n,-1), self.p, 1)
        return torch.mean(d / b)

lp = LpLoss()

## 10. Обучение

In [ ]:
EPOCHS = 300
LR = 1e-3
STEP = 50
W_PRES = 1.0   # вес давления в суммарном loss
W_SAT  = 1.0   # вес насыщенности

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP, gamma=0.5)

os.makedirs(OUT_DIR, exist_ok=True)
PSPAN = NORM['PMAX'] - NORM['PMIN']
SSPAN = NORM['SMAX'] - NORM['SMIN']

h = {'tr':[], 'va':[], 'mae_p':[], 'mae_s':[], 'sk_p':[], 'sk_s':[]}
best_val = float('inf')
t0 = time.time()

for ep in range(EPOCHS):
    model.train()
    tr_loss = 0.0
    for xb, yp, ys in train_loader:
        xb, yp, ys = xb.to(device), yp.to(device), ys.to(device)
        optimizer.zero_grad()
        pp, ps = model(xb)
        loss = W_PRES * lp(pp, yp) + W_SAT * lp(ps, ys)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * xb.size(0)
    tr_loss /= len(train_loader.dataset)

    model.eval()
    va_loss = 0.0
    mae_p, mae_s = 0.0, 0.0
    sse_p, ssm_p, sse_s, ssm_s = 0.0, 0.0, 0.0, 0.0
    with torch.no_grad():
        for xb, yp, ys in val_loader:
            xb, yp, ys = xb.to(device), yp.to(device), ys.to(device)
            pp, ps = model(xb)
            va_loss += (W_PRES * lp(pp, yp) + W_SAT * lp(ps, ys)).item() * xb.size(0)
            mae_p += (torch.abs(pp - yp).mean() * PSPAN).item() * xb.size(0)
            mae_s += (torch.abs(ps - ys).mean() * SSPAN).item() * xb.size(0)

            bp = yp.mean(dim=(1,2), keepdim=True)
            bs = ys.mean(dim=(1,2), keepdim=True)
            sse_p += ((pp - yp)**2).sum().item()
            ssm_p += ((bp - yp)**2).sum().item()
            sse_s += ((ps - ys)**2).sum().item()
            ssm_s += ((bs - ys)**2).sum().item()

    n_va = len(val_loader.dataset)
    va_loss /= n_va; mae_p /= n_va; mae_s /= n_va
    sk_p = 1 - sse_p / max(ssm_p, 1e-12)
    sk_s = 1 - sse_s / max(ssm_s, 1e-12)

    scheduler.step()
    h['tr'].append(tr_loss); h['va'].append(va_loss)
    h['mae_p'].append(mae_p); h['mae_s'].append(mae_s)
    h['sk_p'].append(sk_p); h['sk_s'].append(sk_s)

    if va_loss < best_val:
        best_val = va_loss
        torch.save({'model_state': model.state_dict(), 'norm': NORM,
                    'arch': {'in_ch': IN_CH, 'modes': MODES,
                             'width': WIDTH, 'n_layers': LAYERS},
                    'epoch': ep, 'val_loss': float(va_loss)}, MODEL_PATH)

    if ep % 10 == 0 or ep == EPOCHS - 1:
        print(f'ep {ep:3d} | tr {tr_loss:.4f} | va {va_loss:.4f} '
              f'| MAE_p {mae_p:.2f}б | MAE_s {mae_s:.4f} '
              f'| sk_p {sk_p:.3f} | sk_s {sk_s:.3f}')

print(f'\nобучение: {(time.time()-t0)/60:.1f} мин')
print(f'лучший val: {best_val:.4f} -> {MODEL_PATH}')

## 11. Кривые обучения

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].plot(h['tr'], label='train')
axes[0,0].plot(h['va'], label='val')
axes[0,0].set_yscale('log'); axes[0,0].legend(); axes[0,0].grid(alpha=.3)
axes[0,0].set_title('Суммарный loss'); axes[0,0].set_xlabel('эпоха')

axes[0,1].plot(h['mae_p'], color='tab:blue', label='давление, бар')
axes[0,1].plot(h['mae_s'], color='tab:orange', label='насыщенность')
axes[0,1].legend(); axes[0,1].grid(alpha=.3)
axes[0,1].set_title('MAE'); axes[0,1].set_xlabel('эпоха')

axes[1,0].plot(h['sk_p'], color='tab:blue')
axes[1,0].axhline(0, color='k', lw=.8, ls='--')
axes[1,0].set_ylim(-0.1, 1.0); axes[1,0].grid(alpha=.3)
axes[1,0].set_title('Skill — давление'); axes[1,0].set_xlabel('эпоха')

axes[1,1].plot(h['sk_s'], color='tab:orange')
axes[1,1].axhline(0, color='k', lw=.8, ls='--')
axes[1,1].set_ylim(-0.1, 1.0); axes[1,1].grid(alpha=.3)
axes[1,1].set_title('Skill — насыщенность'); axes[1,1].set_xlabel('эпоха')

plt.tight_layout(); plt.show()

## 12. Визуальная проверка: прогноз vs истина по одной траектории

Берём один сэмпл (все 10 шагов), прогоняем модель, показываем рядом
истинные и предсказанные поля насыщенности и давления.

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

def denorm_p(a): return a * (NORM['PMAX'] - NORM['PMIN']) + NORM['PMIN']
def denorm_s(a): return a * (NORM['SMAX'] - NORM['SMIN']) + NORM['SMIN']

# берём один оригинальный файл
test_file = files[len(files)//2]   # примерно из середины
d = sio.loadmat(test_file)
perm = d['perm_field']
pres_traj = d['pressure_traj']
sat_traj  = d['saturation_traj']
coords = d['well_coords'].astype(int)
wtype  = d['well_type'].astype(int).ravel()

nx, ny = perm.shape
n_steps = pres_traj.shape[2]

inj_pts  = [(I-1, J-1) for (I, J), t in zip(coords, wtype) if t == 1]
prod_pts = [(I-1, J-1) for (I, J), t in zip(coords, wtype) if t != 1]
inj_i,  inj_j  = zip(*inj_pts)  if inj_pts  else ([], [])
prod_i, prod_j = zip(*prod_pts) if prod_pts else ([], [])

perm_n = mm(np.log10(perm), NORM['KMIN'], NORM['KMAX'])
inj_mask  = smooth_mask(inj_pts, nx, ny)
prod_mask = smooth_mask(prod_pts, nx, ny)

show_steps = np.linspace(0, n_steps - 1, min(5, n_steps)).astype(int)

fig, axes = plt.subplots(4, len(show_steps), figsize=(4*len(show_steps), 14))

for col, t in enumerate(show_steps):
    t_norm = t / max(n_steps - 1, 1)
    time_ch = np.full((nx, ny), t_norm)
    x = np.stack([perm_n, inj_mask, prod_mask, time_ch], axis=-1)[None, ...]

    with torch.no_grad():
        pp, ps = model(torch.from_numpy(x).float().to(device))
        pp = pp.cpu().numpy()[0]
        ps = ps.cpu().numpy()[0]

    pred_p = denorm_p(pp)
    pred_s = denorm_s(ps)
    true_p = pres_traj[:, :, t]
    true_s = sat_traj[:, :, t]

    # строка 0: истинная насыщенность
    im = axes[0, col].imshow(true_s.T, origin='lower', cmap='Blues', vmin=0, vmax=1)
    axes[0, col].scatter(inj_i, inj_j, c='red', s=30, edgecolors='k', linewidths=.5)
    axes[0, col].scatter(prod_i, prod_j, c='lime', s=30, edgecolors='k', linewidths=.5)
    axes[0, col].set_title(f'Sw истина, t={t}', fontsize=9)
    plt.colorbar(im, ax=axes[0, col], fraction=.046)

    # строка 1: предсказанная насыщенность
    im = axes[1, col].imshow(pred_s.T, origin='lower', cmap='Blues', vmin=0, vmax=1)
    axes[1, col].scatter(inj_i, inj_j, c='red', s=30, edgecolors='k', linewidths=.5)
    axes[1, col].scatter(prod_i, prod_j, c='lime', s=30, edgecolors='k', linewidths=.5)
    axes[1, col].set_title(f'Sw прогноз, t={t}', fontsize=9)
    plt.colorbar(im, ax=axes[1, col], fraction=.046)

    # строка 2: истинное давление
    im = axes[2, col].imshow(true_p.T, origin='lower', cmap='viridis')
    axes[2, col].set_title(f'P истина, t={t}', fontsize=9)
    plt.colorbar(im, ax=axes[2, col], fraction=.046, label='бар')

    # строка 3: предсказанное давление
    im = axes[3, col].imshow(pred_p.T, origin='lower', cmap='viridis')
    axes[3, col].set_title(f'P прогноз, t={t}', fontsize=9)
    plt.colorbar(im, ax=axes[3, col], fraction=.046, label='бар')

plt.suptitle(f'Прогноз vs истина: {os.path.basename(test_file)}', fontsize=13)
plt.tight_layout(); plt.show()

## 13. Что дальше

Это первая версия двухфазной модели — минимальная рабочая архитектура.
Результаты покажут, где именно модель теряет точность:

- **Если skill по давлению высокий, а по насыщенности низкий** — фронт вытеснения,
  как и ожидали из статей. Следующий шаг: маскированная loss или physics residual
  по сохранению массы.

- **Если оба skill низкие** — скорее всего, 200 сэмплов × 10 шагов недостаточно
  для двух связанных полей. Следующий шаг: увеличение данных.

- **Если train/val расходятся** — переобучение, знакомая история: уменьшать width
  или увеличивать данные.

Возможные улучшения для следующих версий:
- [ ] Перекрёстный гейтинг между ветвями давления и насыщенности (как в DB-AFNO)
- [ ] Time2Vec вместо простого скалярного t/t_max
- [ ] Physics residual: сохранение массы
- [ ] Curriculum по времени: сначала учимся на ранних шагах, потом расширяем